In [18]:
import argparse
# import pyhocon
from tqdm import tqdm
from itertools import combinations
from sklearn.utils import shuffle
from torch.utils import data
import torch

In [50]:


from models import  SpanEmbedder, SpanScorer, FullCrossEncoder
# from evaluator import Evaluation
from dataset import CrossEncoderDatasetFull
from utils import *




In [25]:
# I will create a different config for the dataset and for the training settings

In [26]:
class DataConfig():
    def __init__(self, data_folder,mention_type,use_gold_mentions=True):
        self.data_folder = data_folder
        self.use_gold_mentions= use_gold_mentions
        self.mention_type = mention_type

In [27]:
ecbplus_config = DataConfig(data_folder="ECBplus-prep/output_data/",
                        mention_type="entities_mentions",
                        use_gold_mentions=True)

In [31]:
from collections import namedtuple



In [44]:
train_config = {
  "gpu_num" : [0],
  "bert_model": "roberta-large",
  "roberta_model": "roberta-large",
  "bert_hidden_size": 1024,
  "hidden_layer": 1024,
  "dropout": 0.3,
  "with_mention_width": True,
  "with_head_attention": True,
  "embedding_dimension": 20,

  "max_mention_span": 10,
  "use_gold_mentions": True,
  "mention_type": "events",
  "top_k": 0.25,
  "training_method": "continue",
  "subtopic": True,
  "use_predicted_topics": False,
  "segment": True,

  "random_seed": 0,
  "epochs": 10,
  "batch_size": 32,
  "learning_rate": 1e-4,
  "weight_decay": 0,
  "loss": "bce",
  "optimizer": "adam",
  "adam_epsilon": 1e-8,
  "segment_window": 512,
  "neg_samp": True,
  "exact": False,

  "log_path": "logs/pairwise_scorer/",
  "data_folder": "data/ecb/mentions",
  "span_repr_path": "models/span_scorers/events_span_repr_0",
  "span_scorer_path": "models/span_scorers/events_span_scorer_0",
  "model_path": "models/pairwise_scorers"
}

In [47]:
ConfigTuple = namedtuple('TrainConfig', train_config)

In [48]:
config = ConfigTuple(**train_config)

In [9]:
train = CrossEncoderDatasetFull(ecbplus_config)

In [35]:
train_loader = data.DataLoader(train, batch_size=config.batch_size, shuffle=True)

In [ ]:
#Take it from here, create class with all the attributes from the json file

In [40]:
device_ids = config.gpu_num
#device = torch.device("cuda:{}".format(device_ids[0]))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [41]:
## Models' initiation
#logger.info('Init models')
span_repr = SpanEmbedder(config, device).to(device)


In [42]:
span_scorer = SpanScorer(config).to(device)

In [49]:
cross_encoder_single = FullCrossEncoder(config).to(device)

Downloading:   0%|          | 0.00/899k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/456k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/482 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

In [51]:
cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=device_ids)

In [52]:
## Optimizer and loss function
criterion = get_loss_function(config)
optimizer = get_optimizer(config, [cross_encoder])
scheduler = get_scheduler(optimizer, total_steps=config.epochs * len(train_loader))

In [56]:
f1 = []
for epoch in range(config.epochs):
    # logger.info('Epoch: {}'.format(epoch))
    accumulate_loss = 0
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.train()

    for batch_x, batch_y in tqdm(train_loader):
        bert_tokens = cross_encoder.module.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        optimizer.zero_grad()
        cross_encoder.zero_grad()
        scores = cross_encoder(input_ids, attention_mask)
        loss = criterion(scores, batch_y.to(device))
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(cross_encoder.parameters(), 1.0)
        optimizer.step()
        scheduler.step()


        accumulate_loss += loss.item()

        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)

  0%|          | 0/42533 [00:00<?, ?it/s]c:\ProgramData\miniconda3\envs\cdlm\lib\site-packages\transformers\tokenization_utils_base.py:1773: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,
  0%|          | 1/42533 [05:40<4024:29:38, 340.64s/it]


KeyboardInterrupt: 